# Laboratorio 2: Adquisición Programática y Almacenamiento de Datos Masivos desde la API de CoinGecko
**Asignatura:** Big Data / Ciencia de Datos  
**Fecha:** 1 de septiembre de 2026

**Integrantes:**

Jhonatan Smith Doncell

Jorge Emmanuel Marin

Adrian Felipe Grateron


En la presente práctica de laboratorio se diseña e implementa un flujo de trabajo (pipeline) para la adquisición, procesamiento y almacenamiento de datos masivos provenientes de una interfaz de programación de aplicaciones (API) REST externa. Se capturan series de tiempo de precios históricos de criptomonedas públicas a través de la API de CoinGecko, aplicando patrones de ingesta semiestructurada (JSON). Finalmente, se implementan principios de arquitectura Lakehouse para la persistencia multicapa mediante formatos tabulares (CSV, Parquet) y una capa relacional y transaccional basada en Delta Lake sobre Databricks.

### Introducción y Contexto

En el presente laboratorio se diseña e implementa un flujo de ingeniería de datos (*pipeline*) para la adquisición y almacenamiento de información financiera desde una API REST externa. Mediante el consumo de datos públicos de criptomonedas provenientes de la plataforma CoinGecko, se estructuran capas de procesamiento progresivo alineadas con los principios de organización y persistencia de una arquitectura tipo **Lakehouse**.

## Objetivos de la Práctica

Al finalizar el laboratorio, se habrán alcanzado los siguientes objetivos técnicos:

1. **Consumo programático:** Interprocesar y extraer datos desde una API REST pública.
2. **Identificación de estructuras:** Analizar e interpretar esquemas semiestructurados en formato JSON.
3. **Transformación de datos:** Normalizar y convertir estructuras JSON a esquemas tabulares.
4. **Persistencia multicapa:** Almacenar datos procesados en formatos CSV, Parquet y Delta.
5. **Modelado en Databricks:** Registrar y gestionar tablas Delta en el catálogo de Unity/Workspace.
6. **Transaccionalidad:** Ejecutar operaciones de lectura/escritura con propiedades ACID sobre Delta Lake.

## 1. Preparación y Configuración del Entorno de Ejecución

En esta fase se inicializan las dependencias requeridas (`requests`, `pandas`, `pyspark`), se definen las constantes globales del laboratorio y se asegura la existencia del esquema y volumen dentro de la arquitectura de almacenamiento de Databricks.

In [ ]:
#Configuración general del laboratorio
import requests
import time
import json
from datetime import datetime, date, UTC
import pandas as pd
 
from pyspark.sql import functions as F
from pyspark.sql import Row
 
CATALOG = "workspace"
SCHEMA = "bigdata"
VOLUME = "lab_files"
 
TABLE_NAME = "coin_prices_lab2"
FULL_TABLE_NAME = f"{CATALOG}.{SCHEMA}.{TABLE_NAME}"
 
COINS = ["bitcoin", "ethereum", "solana"]
VS_CURRENCY = "usd"
DAYS = 60
 
# Timestamp UTC con zona horaria
RUN_TS = datetime.now(UTC)
 
# String segura para rutas
RUN_DATE = RUN_TS.strftime("%Y-%m-%d")
 
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.{SCHEMA}.{VOLUME}")
 
RAW_BASE_PATH = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}/lab2/raw/{RUN_DATE}"
PROCESSED_BASE_PATH = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}/lab2/processed/{RUN_DATE}"
 
CSV_OUTPUT_PATH = f"{PROCESSED_BASE_PATH}/csv"
PARQUET_OUTPUT_PATH = f"{PROCESSED_BASE_PATH}/parquet"
 
print("Tabla destino:", FULL_TABLE_NAME)
print("Fecha de ejecución:", RUN_DATE)
print("RAW_BASE_PATH:", RAW_BASE_PATH)
print("PROCESSED_BASE_PATH:", PROCESSED_BASE_PATH)
print(f"Esquema listo: {CATALOG}.{SCHEMA}")
print(f"Volumen listo: {CATALOG}.{SCHEMA}.{VOLUME}")

## 2. Consulta y Extracción desde la API de CoinGecko

Se utiliza el punto de enlace (*endpoint*) `/coins/{id}/market_chart` con el objetivo de consultar series temporales de precios históricos para criptomonedas seleccionadas.

* **Documentación oficial:** [CoinGecko API v3 Documentation](https://www.coingecko.com/en/api/documentation)

In [ ]:
def fetch_market_chart(
    coin: str,
    vs_currency: str = "usd",
    days: int = 60,
    max_retries: int = 4
) -> dict:
    """Consulta precios históricos desde CoinGecko con reintentos básicos."""
    url = f"https://api.coingecko.com/api/v3/coins/{coin}/market_chart"
    params = {
        "vs_currency": vs_currency,
        "days": days
    }
 
    response = None
 
    for attempt in range(max_retries):
        response = requests.get(url, params=params, timeout=30)
 
        if response.status_code == 200:
            return response.json()
 
        if response.status_code in (429, 500, 502, 503, 504):
            wait_time = 2 * (attempt + 1)
            print(f"Reintento {attempt + 1}/{max_retries} para {coin}. Esperando {wait_time}s...")
            time.sleep(wait_time)
        else:
            response.raise_for_status()

## 3. Adquisición y Almacenamiento en Capa RAW (JSON)

Los datos crudos obtenidos directamente de la API REST se persisten en la ruta `/raw/` bajo el formato JSON. La conservación de esta capa permite mantener la trazabilidad e inmutabilidad de la información en su estructura semiestructurada original.

In [ ]:
# Extracción de precios históricos por criptomoneda
# Guarda la respuesta en JSON (RAW) y transforma los datos a DataFrame
# Incluye pausas (sleep) y validaciones para evitar errores por límite de peticiones (Rate Limit)
all_processed = []

for coin in COINS:
    print(f"Consultando datos para: {coin}")
    data = fetch_market_chart(coin=coin, vs_currency=VS_CURRENCY, days=DAYS)

    # Validar que la API no haya retornado None
    if data is None:
        print(
            f"No se obtuvieron datos para {coin} (posible error de API o rate limit). Saltando..."
        )
        # Pausa antes de la siguiente moneda para dar tiempo al servidor
        time.sleep(6)
        continue

    raw_path = f"{RAW_BASE_PATH}/{coin}.json"
    dbutils.fs.put(raw_path, json.dumps(data), overwrite=True)
    print(f"RAW guardado en: {raw_path}")

    prices = data.get("prices", []) if isinstance(data, dict) else []

    if not prices:
        print(f"No se encontraron precios para {coin}")
        # Pausa antes de pasar a la siguiente moneda
        time.sleep(6)
        continue

    pdf = pd.DataFrame(prices, columns=["timestamp_ms", "price"])
    pdf["timestamp"] = pd.to_datetime(pdf["timestamp_ms"], unit="ms", utc=True)
    pdf["date"] = pdf["timestamp"].dt.date
    pdf["coin"] = coin
    pdf["vs_currency"] = VS_CURRENCY

    pdf = pdf[["coin", "vs_currency", "timestamp", "date", "price"]]
    all_processed.append(pdf)

    # Pausa respetuosa entre llamadas exitosas para evitar rate limits
    print("Esperando 6 segundos para la siguiente consulta...")
    time.sleep(6)

print("Extracción completada.")

*Inspección de metadatos de los archivos JSON generados en la capa RAW (ubicación, nombre y tamaño en bytes).*

In [ ]:
print("Archivos RAW generados:")
display(dbutils.fs.ls(RAW_BASE_PATH))

## 4. Transformación y Estructuración Tabular (Capa Procesada)

En este paso, los datos semiestructurados (JSON) son procesados para conformar una representación tabular rígida mediante columnas explícitas (`coin`, `vs_currency`, `timestamp`, `date`, `price`).

In [ ]:
df_pd = pd.concat(all_processed, ignore_index=True)
df_pd.head()

*Conversión del DataFrame de Pandas a un DataFrame de PySpark para habilitar el procesamiento distribuido y la definición explícita de tipos de datos.*

In [ ]:
df_spark = spark.createDataFrame(df_pd)
 
df_spark = (
    df_spark
    .withColumn("timestamp", F.col("timestamp").cast("timestamp"))
    .withColumn("date", F.col("date").cast("date"))
    .withColumn("price", F.col("price").cast("double"))
)
 
display(df_spark)

Verificación de los registros y monedas

In [ ]:
df_spark.printSchema()
print(f"Total de registros: {df_spark.count()}")
print(f"Total de monedas: {df_spark.select('coin').distinct().count()}")

##5. Almacenamiento en formatos CSV y parquet
Se almacena la versión procesada en dos formatos físicos:

CSV: Formato plano y ampliamente interoperable.

Parquet: Formato columnar optimizado para analítica.

La versión Delta se almacenará como tabla administrada dentro del catálogo.

In [ ]:
#Creación formato CSV
(
    df_spark.coalesce(1)
    .write
    .mode("overwrite")
    .option("header", True)
    .csv(CSV_OUTPUT_PATH)
)
 
print(f"CSV guardado en: {CSV_OUTPUT_PATH}")

In [ ]:
#Creación formato Parquet
(
    df_spark.write
    .mode("overwrite")
    .parquet(PARQUET_OUTPUT_PATH)
)
 
print(f"Parquet guardado en: {PARQUET_OUTPUT_PATH}")

In [ ]:
print("contenido de processed:")
display(dbutils.fs.ls(PROCESSED_BASE_PATH))

##6. Lectura y comparación inicial de formatos
Se leerán nuevamente los archivos generados para verificar su contenido y comparar su estructura.

In [ ]:
df_csv = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(CSV_OUTPUT_PATH)
)
 
df_parquet = spark.read.parquet(PARQUET_OUTPUT_PATH)
 
print("Registros en CSV:", df_csv.count())
print("Registros en Parquet:", df_parquet.count())
 
print("Esquema CSV")
df_csv.printSchema()
 
print("Esquema Parquet")
df_parquet.printSchema()

##7. Creación de la tabla Delta en el catálogo
Se almacenará la versión principal del dataset como una tabla Delta administrada por Unity Catalog.

(Las tablas de Databricks son tipo Delta)

In [ ]:
spark.sql(f"DROP TABLE IF EXISTS {FULL_TABLE_NAME}")
 
(
    df_spark.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(FULL_TABLE_NAME)
)
 
print(f"Tabla Delta creada: {FULL_TABLE_NAME}")

Validamos el tamaño de cada uno de los archivos

In [ ]:
def get_directory_size_bytes(path: str) -> int:
    """Suma recursivamente el tamaño de todos los archivos dentro de una ruta."""
    total_size = 0
 
    for item in dbutils.fs.ls(path):
        if item.isDir():
            total_size += get_directory_size_bytes(item.path)
        else:
            total_size += item.size
 
    return total_size
 
 
def bytes_to_human_readable(num_bytes: int) -> str:
    """Convierte bytes a una unidad legible."""
    units = ["B", "KB", "MB", "GB", "TB"]
    size = float(num_bytes)
 
    for unit in units:
        if size < 1024 or unit == units[-1]:
            return f"{size:.2f} {unit}"
        size /= 1024
 
 
json_size = get_directory_size_bytes(RAW_BASE_PATH)
csv_size = get_directory_size_bytes(CSV_OUTPUT_PATH)
parquet_size = get_directory_size_bytes(PARQUET_OUTPUT_PATH)
DELTA_COMPARE_PATH = f"{PROCESSED_BASE_PATH}/delta_compare"
(
    df_spark.write
    .format("delta")
    .mode("overwrite")
    .save(DELTA_COMPARE_PATH)
)
delta_size = get_directory_size_bytes(DELTA_COMPARE_PATH)
 
print("Tamaño Delta:", bytes_to_human_readable(delta_size))
print("Tamaño JSON RAW:", bytes_to_human_readable(json_size))
print("Tamaño CSV:", bytes_to_human_readable(csv_size))
print("Tamaño Parquet:", bytes_to_human_readable(parquet_size))

##8. Operaciones ACID básicas sobre Delta Lake
Se realizarán operaciones de inserción, actualización, borrado y combinación (MERGE) para evidenciar el soporte transaccional del formato Delta.

In [ ]:
print("Historial inicial de la tabla Delta:")
display(spark.sql(f"DESCRIBE HISTORY {FULL_TABLE_NAME}"))

Aplicamos una operación de tipo insert para crear dos columnas con la data y precios ficticios (nuevos registros)

In [ ]:
# INSERT / APPEND
 
new_rows = [
    Row(
        coin="bitcoin",
        vs_currency="usd",
        timestamp=datetime.now(UTC),
        date=date.today(),
        price=99999.99
    ),
    Row(
        coin="ethereum",
        vs_currency="usd",
        timestamp=datetime.now(UTC),
        date=date.today(),
        price=4999.99
    )
]
 
new_df = spark.createDataFrame(new_rows)
 
(
    new_df.write
    .format("delta")
    .mode("append")
    .saveAsTable(FULL_TABLE_NAME)
)
 
print("Nuevos registros insertados.")

Visualizamos los ultimos 10 datos de la tabla (validamos los 2 datos insertados)

In [ ]:
display(
    spark.sql(f"""
    SELECT *
    FROM {FULL_TABLE_NAME}
    ORDER BY timestamp DESC
    LIMIT 10
    """)
)

In [ ]:
# UPDATE
 
spark.sql(f"""
UPDATE {FULL_TABLE_NAME}
SET price = price * 1.01
WHERE coin = 'bitcoin'
""")
 
print("UPDATE ejecutado sobre registros de bitcoin.")

In [ ]:
# DELETE
 
spark.sql(f"""
DELETE FROM {FULL_TABLE_NAME}
WHERE price < 0
""")
 
print("DELETE ejecutado.")

In [ ]:
# MERGE
 
updates_data = [
    ("bitcoin", "usd", datetime.now(UTC), date.today(), 88888.88),
    ("cardano", "usd", datetime.now(UTC), date.today(), 0.75)
]
 
updates_pd = pd.DataFrame(
    updates_data,
    columns=["coin", "vs_currency", "timestamp", "date", "price"]
)
 
updates_spark = spark.createDataFrame(updates_pd)
updates_spark.createOrReplaceTempView("coin_updates")
 
spark.sql(f"""
MERGE INTO {FULL_TABLE_NAME} AS target
USING coin_updates AS source
ON target.coin = source.coin AND target.date = source.date
WHEN MATCHED THEN
  UPDATE SET
    target.vs_currency = source.vs_currency,
    target.timestamp = source.timestamp,
    target.price = source.price
WHEN NOT MATCHED THEN
  INSERT (coin, vs_currency, timestamp, date, price)
  VALUES (source.coin, source.vs_currency, source.timestamp, source.date, source.price)
""")
 
print("MERGE ejecutado.")

Podemos recuperar con la metadata las versiones anteriores.

In [ ]:
spark.read.format("delta").option("versionAsOf", 0).table(FULL_TABLE_NAME).count()

In [ ]:
print("Historial actual de la tabla Delta:")
display(spark.sql(f"DESCRIBE HISTORY {FULL_TABLE_NAME}"))

##9. Explica las diferencias entre almacenar la información en JSON, CSV, Parquet y Delta.
##10. Justifica en qué escenarios convendría usar cada formato dentro de un flujo de Big Data.


**CSV:** Plano y legible por humanos, pero ineficiente y sin tipos de datos estrictos.

**JSON:** Flexible para datos anidados y semiestructurados, pero pesado y lento de procesar a gran escala.

**Parquet:** Ultra rápido para consultas analíticas masivas por columnas, pero rígido e inmutable.

**Delta:** Parquet mejorado con transacciones ACID, viajes en el tiempo y MERGE (upserts), pero añade sobrecoste de metadatos.

- Adoptamos CSV exclusivamente en los puntos de salida finales para exportar subconjuntos de datos ligeros hacia herramientas de oficina (como Excel) o para integraciones con sistemas heredados que exijan este formato.

- Utilizamos JSON en la capa de ingesta (Bronze) para capturar eventos y fuentes en tiempo real (APIs, Kafka, IoT), aprovechando su flexibilidad para asimilar cambios o variaciones inmediatas en la estructura de los datos.

- Reservamos Parquet para repositorios de almacenamiento frío o datasets estáticos donde solo realizamos lecturas analíticas masivas e históricas y donde garantizamos que no habrá modificaciones de filas.

- Establecemos Delta Lake como nuestro estándar predeterminado en las capas del Lakehouse (Bronze, Silver y Gold) para todos los pipelines analíticos activos, asegurando soporte para mutaciones de datos (MERGE/upserts), cumplimiento de borrados normativos (GDPR) e ingestas concurrentes sin riesgo de corrupción.

##11. Reflexiona sobre los desafíos y decisiones tomadas durante la práctica:

- **¿Qué diferencias encontraste entre los datos semiestructurados y los estructurados?**

Los datos semiestructurados permiten manejar esquemas altamente cambiantes, campos opcionales y estructuras anidadas complejas sin necesidad de definir un esquema rígido previo.
Por otro lado los datos estructurados, presentan un menor tamaño, facilidad de usos en analítica y en Delta transacciones ACID.

- **¿Qué formato consideras más adecuado para el análisis de datos y por qué?**

Consideramos que el formato Delta tiene un mayor potencial de uso, debido a sus viajes entre versiones, su bajo peso y particiones.

- **¿Qué ventajas ofrece Delta Lake frente a otros formatos de almacenamiento?**

Transacciones ACID, soporte para DML (Delete, Merge y Update), Timetravel y auditoría por sus versiones.

- **¿Por qué son importantes las propiedades ACID en un entorno moderno de datos?**

Porque permiten trabajar cambios sobre versiones, facilitan la reutilización de las bases de datos y nos ayudan a tener trazabilidad de las modificaciones.

- **¿Qué problemas surgieron al adquirir datos desde una API externa y cómo podrían mitigarse en un escenario real?**

Aveces al adquirir datos de una API externa, los intentos pueden fallar y no extraer la totalidad de los datos solicitados, se podrían mitigar en un escenario real descargando por lotes o respetanto Time Sleep.

---

## Referencias Bibliográficas

Apache Software Foundation. (2023). *Apache Spark: Unified analytics engine for large-scale data processing* (Versión 3.5.0) [Software]. https://spark.apache.org/

CoinGecko. (2024). *CoinGecko API v3 documentation: Public REST API for crypto market data*. https://www.coingecko.com/en/api/documentation

Databricks. (2024). *What is Delta Lake in Databricks? Databricks Documentation*. https://docs.databricks.com/en/delta/index.html

McKinney, W. (2010). Data structures for statistical computing in Python. En S. van der Walt & J. Millman (Eds.), *Proceedings of the 9th Python in Science Conference* (pp. 56–61). https://doi.org/10.25080/Majora-92bf192d-00a

The pandas Development Team. (2024). *pandas-dev/pandas: pandas* (Versión 2.2.0) [Software]. Zenodo. https://doi.org/10.5281/zenodo.3509134